In [ ]:
import duckdb
from pathlib import Path

project_path = Path.cwd()
print("Current folder:", project_path)
print("DuckDB version:", duckdb.__version__)

In [ ]:
import duckdb

source_db_path = project_path / "05_Starter" / "pulseiq.duckdb"

con = duckdb.connect(str(source_db_path), read_only=True)

tables_df = con.execute("SHOW ALL TABLES").fetchdf()

print("Source database tables:")
display(tables_df)

In [ ]:
metric_source_tables = [
    "fact_headcount_monthly",
    "dim_employee",
    "fact_payroll",
    "fact_attendance_monthly",
    "fact_leave",
    "fact_ledger",
    "fact_invoice",
    "fact_invoice_payment",
    "fact_task",
    "dim_sprint",
    "fact_anomaly",
    "fact_attrition_risk",
    "fact_forecast",
    "dim_department",
    "dim_team",
    "dim_account",
    "dim_date",
]

for table in metric_source_tables:
    print(f"\n{'=' * 65}\n{table}")
    display(
        con.execute(f"DESCRIBE SELECT * FROM raw.{table}").fetchdf()
    )

In [ ]:
import pandas as pd
from pathlib import Path

dictionary_path = project_path / "03_Data" / "csv" / "metric_dictionary.csv"

metric_dictionary = pd.read_csv(dictionary_path)

implementation_map = metric_dictionary[
    [
        "metric_id",
        "metric_name",
        "formula",
        "source_table",
        "grain",
        "minimum_group_size",
        "caveat",
    ]
].copy()

display(implementation_map)
print("Total official metrics:", len(implementation_map))
print("Unique metric IDs:", implementation_map["metric_id"].nunique())

In [ ]:
warehouse_db_path = project_path / "05_Starter" / "pulseiq_warehouse.duckdb"

print("Warehouse database exists:", warehouse_db_path.exists())

warehouse_con = duckdb.connect(
    str(warehouse_db_path),
    read_only=True
)

warehouse_tables = warehouse_con.execute(
    "SHOW ALL TABLES"
).fetchdf()

display(warehouse_tables)

In [ ]:
print("Existing Python variables:")
print("con:", "available" if "con" in globals() else "not found")
print(
    "warehouse_con:",
    "available" if "warehouse_con" in globals() else "not found"
)

print("\nWarehouse file exists:", warehouse_db_path.exists())

In [ ]:
import os

print("PQ-04 process ID:", os.getpid())
print("Warehouse path:", warehouse_db_path)
print("Warehouse exists:", warehouse_db_path.exists())

In [ ]:
print("Source connection:", con.execute("SELECT 1").fetchone())

print("Warehouse connection variable exists:",
      "warehouse_con" in globals())

In [ ]:
import os
from pathlib import Path

print("Current kernel PID:", os.getpid())
print("Warehouse file:", warehouse_db_path)
print("Warehouse file size:", warehouse_db_path.stat().st_size, "bytes")

In [ ]:
print(con.execute("PRAGMA database_list").fetchdf())

In [ ]:
import duckdb

warehouse_db_path = project_path / "05_Starter" / "pulseiq_warehouse.duckdb"

warehouse_con = duckdb.connect(
    str(warehouse_db_path),
    read_only=True
)

print("Warehouse connection successful!")
print(warehouse_con.execute("PRAGMA database_list").fetchdf())

In [ ]:
warehouse_tables = warehouse_con.execute("""
    SELECT table_schema, table_name, table_type
    FROM information_schema.tables
    ORDER BY table_schema, table_name
""").fetchdf()

display(warehouse_tables)

In [ ]:
existing_views = warehouse_con.execute("""
    SELECT table_schema, table_name
    FROM information_schema.views
    ORDER BY table_schema, table_name
""").fetchdf()

print("Existing views:", len(existing_views))
display(existing_views)

In [ ]:
# Check the warehouse tables needed for Task 4
warehouse_con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'warehouse'
      AND table_name IN (
          'fact_headcount_monthly',
          'fact_payroll',
          'fact_attendance_monthly',
          'fact_ledger',
          'fact_invoice',
          'fact_task',
          'fact_anomaly',
          'fact_attrition_risk',
          'fact_forecast',
          'dim_employee',
          'dim_department',
          'dim_team',
          'dim_sprint',
          'dim_account'
      )
    ORDER BY table_name
""").fetchdf()

In [ ]:
metric_dictionary_path = (
    project_path / "03_Data" / "csv" / "metric_dictionary.csv"
)

import pandas as pd

metric_df = pd.read_csv(metric_dictionary_path)

print("Total metrics:", len(metric_df))
print("Unique metric IDs:", metric_df["metric_id"].nunique())

display(
    metric_df[
        [
            "metric_id",
            "metric_name",
            "source_table",
            "grain",
            "minimum_group_size"
        ]
    ].sort_values("metric_id")
)

In [ ]:
display(
    metric_df[
        [
            "metric_id",
            "metric_name",
            "business_definition",
            "formula",
            "source_table",
            "grain",
            "caveat",
            "minimum_group_size"
        ]
    ].sort_values("metric_id").to_string(index=False)
)

In [ ]:
example_sql_path = (
    project_path / "05_Starter" / "metric_views_examples.sql"
)

print("File exists:", example_sql_path.exists())

if example_sql_path.exists():
    print(example_sql_path.read_text(encoding="utf-8"))

In [ ]:
tables_to_inspect = [
    "fact_headcount_monthly",
    "dim_employee",
    "fact_payroll",
]

for table_name in tables_to_inspect:
    print(f"\n--- warehouse.{table_name} ---")
    columns = warehouse_con.execute(f"""
        DESCRIBE warehouse.{table_name}
    """).fetchdf()
    display(columns[["column_name", "column_type"]])

In [ ]:
tables_to_inspect = [
    "fact_attendance_monthly",
    "fact_ledger",
    "fact_invoice",
    "fact_task",
]

for table_name in tables_to_inspect:
    print(f"\n--- warehouse.{table_name} ---")
    columns = warehouse_con.execute(
        f"DESCRIBE warehouse.{table_name}"
    ).fetchdf()
    display(columns[["column_name", "column_type"]])

In [ ]:
tables_to_inspect = [
    "fact_anomaly",
    "fact_attrition_risk",
    "fact_forecast",
    "dim_team",
    "dim_sprint",
    "dim_department",
    "dim_account",
]

for table_name in tables_to_inspect:
    print(f"\n--- warehouse.{table_name} ---")
    columns = warehouse_con.execute(
        f"DESCRIBE warehouse.{table_name}"
    ).fetchdf()
    display(columns[["column_name", "column_type"]])

In [ ]:
payroll_quality = warehouse_con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT payroll_item_id) AS unique_payroll_items,
        COUNT(*) - COUNT(DISTINCT payroll_item_id) AS duplicate_payroll_item_ids,
        SUM(CASE WHEN headcount_flag IS NULL THEN 1 ELSE 0 END)
            AS null_headcount_flags,
        SUM(CASE WHEN headcount_flag = 0 THEN 1 ELSE 0 END)
            AS zero_headcount_flags,
        SUM(CASE WHEN headcount_flag = 1 THEN 1 ELSE 0 END)
            AS one_headcount_flags
    FROM warehouse.fact_payroll
""").fetchdf()

display(payroll_quality)

In [ ]:
payroll_counts = warehouse_con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT payroll_item_id) AS unique_payroll_items,
        COUNT(*) FILTER (
            WHERE headcount_flag = 0
        ) AS zero_headcount_flags,
        COUNT(*) FILTER (
            WHERE headcount_flag = 1
        ) AS one_headcount_flags
    FROM warehouse.fact_payroll
""").fetchone()

print("Total rows:", payroll_counts[0])
print("Unique payroll item IDs:", payroll_counts[1])
print("Zero headcount flags:", payroll_counts[2])
print("One headcount flags:", payroll_counts[3])

In [ ]:
payroll_anomalies = warehouse_con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT payroll_item_id) AS unique_payroll_items,
        COUNT(*) - COUNT(DISTINCT employee_id || '|' || run_id)
            AS excess_employee_run_rows
    FROM warehouse.fact_payroll
""").fetchdf()

display(payroll_anomalies)

In [ ]:
duplicate_employee_runs = warehouse_con.execute("""
    SELECT
        employee_id,
        run_id,
        COUNT(*) AS row_count,
        STRING_AGG(payroll_item_id, ', ') AS payroll_item_ids
    FROM warehouse.fact_payroll
    GROUP BY employee_id, run_id
    HAVING COUNT(*) > 1
    ORDER BY employee_id, run_id
""").fetchdf()

print("Duplicate employee-run keys:", len(duplicate_employee_runs))
display(duplicate_employee_runs)


In [ ]:
orphan_payroll = warehouse_con.execute("""
    SELECT
        p.payroll_item_id,
        p.employee_id,
        p.run_id,
        p.period_end_date,
        p.department_id
    FROM warehouse.fact_payroll p
    LEFT JOIN warehouse.dim_employee e
        ON p.employee_id = e.employee_id
    WHERE e.employee_id IS NULL
    ORDER BY p.payroll_item_id
""").fetchdf()

print("Orphan payroll rows:", len(orphan_payroll))
display(orphan_payroll)

In [ ]:
warehouse_con.execute("""
    SELECT
        payroll_item_id,
        employee_id,
        run_id,
        period_end_date,
        department_id,
        gross_pay,
        net_pay,
        employer_cost
    FROM warehouse.fact_payroll
    WHERE payroll_item_id IN (
        'PI-004059', 'PI-X0001',
        'PI-000761', 'PI-X0002',
        'PI-X0003', 'PI-X0004'
    )
    ORDER BY employee_id, run_id, payroll_item_id
""").fetchdf()

In [ ]:
warehouse_con.close()
print("Read-only warehouse connection closed.")

In [ ]:
warehouse_con = duckdb.connect(
    str(warehouse_db_path),
    read_only=False
)

print("Writable connection opened successfully.")
print(
    warehouse_con.execute(
        "PRAGMA database_list"
    ).fetchdf()
)

In [ ]:
warehouse_con.execute("""
CREATE OR REPLACE VIEW warehouse.clean_payroll AS
SELECT p.*
FROM warehouse.fact_payroll AS p
WHERE p.payroll_item_id NOT IN (
    'PI-X0001',
    'PI-X0002',
    'PI-X0003',
    'PI-X0004'
)
""")

print("Clean-payroll view created successfully.")

print(
    warehouse_con.execute("""
        SELECT
            COUNT(*) AS clean_rows,
            COUNT(DISTINCT employee_id || '|' || run_id)
                AS unique_employee_run_keys
        FROM warehouse.clean_payroll
    """).fetchdf()
)

In [ ]:
warehouse_con.execute("""
CREATE OR REPLACE VIEW warehouse.m01_headcount AS
SELECT
    h.month_start_date,
    h.department_id,
    COUNT(DISTINCT e.employee_id) AS headcount
FROM warehouse.fact_headcount_monthly AS h
JOIN warehouse.dim_employee AS e
    ON e.department_id = h.department_id
   AND e.hire_date <= (
       h.month_start_date + INTERVAL '1 month' - INTERVAL '1 day'
   )
   AND (
       e.exit_date IS NULL
       OR e.exit_date > (
           h.month_start_date + INTERVAL '1 month' - INTERVAL '1 day'
       )
   )
GROUP BY h.month_start_date, h.department_id;

CREATE OR REPLACE VIEW warehouse.m02_hires AS
SELECT
    month_start_date,
    department_id,
    hires
FROM warehouse.fact_headcount_monthly;

CREATE OR REPLACE VIEW warehouse.m03_exits AS
SELECT
    month_start_date,
    department_id,
    exits
FROM warehouse.fact_headcount_monthly;
""")

print("Created metric views: M01, M02, M03")

In [ ]:
metric_check = warehouse_con.execute("""
    SELECT
        'M01 Headcount' AS metric,
        COUNT(*) AS result_rows,
        SUM(CASE WHEN headcount < 5 THEN 1 ELSE 0 END)
            AS groups_below_minimum
    FROM warehouse.m01_headcount

    UNION ALL

    SELECT
        'M02 Hires',
        COUNT(*),
        SUM(CASE WHEN hires < 0 THEN 1 ELSE 0 END)
    FROM warehouse.m02_hires

    UNION ALL

    SELECT
        'M03 Exits',
        COUNT(*),
        SUM(CASE WHEN exits < 0 THEN 1 ELSE 0 END)
    FROM warehouse.m03_exits
""").fetchdf()

display(metric_check)

In [ ]:
warehouse_con.execute("""
CREATE OR REPLACE VIEW warehouse.m01_headcount_governed AS
SELECT
    month_start_date,
    department_id,
    CASE
        WHEN headcount >= 5 THEN headcount
        ELSE NULL
    END AS headcount,
    CASE
        WHEN headcount >= 5 THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM warehouse.m01_headcount
""")

print("Governed headcount view created.")

display(warehouse_con.execute("""
    SELECT
        privacy_status,
        COUNT(*) AS group_count
    FROM warehouse.m01_headcount_governed
    GROUP BY privacy_status
    ORDER BY privacy_status
""").fetchdf())

In [ ]:
warehouse_con.execute("""
CREATE OR REPLACE VIEW warehouse.m04_attrition_monthly AS
SELECT
    month_start_date,
    department_id,
    CASE
        WHEN headcount_start >= 10
        THEN ROUND(100.0 * exits / NULLIF(headcount_start, 0), 2)
        ELSE NULL
    END AS attrition_rate_pct,
    CASE
        WHEN headcount_start >= 10 THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM warehouse.fact_headcount_monthly;

CREATE OR REPLACE VIEW warehouse.m05_attrition_rolling_12m AS
WITH rolling AS (
    SELECT
        month_start_date,
        department_id,
        SUM(exits) OVER (
            PARTITION BY department_id
            ORDER BY month_start_date
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS exits_12m,
        AVG(headcount_end) OVER (
            PARTITION BY department_id
            ORDER BY month_start_date
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS avg_headcount_12m,
        COUNT(*) OVER (
            PARTITION BY department_id
            ORDER BY month_start_date
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS months_available
    FROM warehouse.fact_headcount_monthly
)
SELECT
    month_start_date,
    department_id,
    CASE
        WHEN months_available = 12
         AND avg_headcount_12m >= 10
        THEN ROUND(100.0 * exits_12m / NULLIF(avg_headcount_12m, 0), 2)
        ELSE NULL
    END AS attrition_rate_12m_pct,
    CASE
        WHEN months_available = 12
         AND avg_headcount_12m >= 10
        THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM rolling;

CREATE OR REPLACE VIEW warehouse.m06_voluntary_attrition AS
WITH rolling AS (
    SELECT
        month_start_date,
        department_id,
        SUM(voluntary_exits) OVER (
            PARTITION BY department_id
            ORDER BY month_start_date
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS voluntary_exits_12m,
        AVG(headcount_end) OVER (
            PARTITION BY department_id
            ORDER BY month_start_date
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS avg_headcount_12m,
        COUNT(*) OVER (
            PARTITION BY department_id
            ORDER BY month_start_date
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS months_available
    FROM warehouse.fact_headcount_monthly
)
SELECT
    month_start_date,
    department_id,
    CASE
        WHEN months_available = 12
         AND avg_headcount_12m >= 10
        THEN ROUND(
            100.0 * voluntary_exits_12m / NULLIF(avg_headcount_12m, 0),
            2
        )
        ELSE NULL
    END AS voluntary_attrition_rate_pct,
    CASE
        WHEN months_available = 12
         AND avg_headcount_12m >= 10
        THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM rolling;
""")

print("Created M04, M05, and M06.")

In [ ]:
attrition_validation = warehouse_con.execute("""
    SELECT
        'M04 Monthly attrition' AS metric,
        COUNT(*) AS total_groups,
        COUNT(*) FILTER (
            WHERE privacy_status = 'PUBLISHED'
        ) AS published_groups,
        COUNT(*) FILTER (
            WHERE privacy_status = 'SUPPRESSED'
        ) AS suppressed_groups,
        COUNT(*) FILTER (
            WHERE attrition_rate_pct IS NOT NULL
              AND (
                  attrition_rate_pct < 0
                  OR attrition_rate_pct > 100
              )
        ) AS invalid_values
    FROM warehouse.m04_attrition_monthly

    UNION ALL

    SELECT
        'M05 Rolling 12m attrition',
        COUNT(*),
        COUNT(*) FILTER (WHERE privacy_status = 'PUBLISHED'),
        COUNT(*) FILTER (WHERE privacy_status = 'SUPPRESSED'),
        COUNT(*) FILTER (
            WHERE attrition_rate_12m_pct IS NOT NULL
              AND attrition_rate_12m_pct < 0
        )
    FROM warehouse.m05_attrition_rolling_12m

    UNION ALL

    SELECT
        'M06 Voluntary attrition',
        COUNT(*),
        COUNT(*) FILTER (WHERE privacy_status = 'PUBLISHED'),
        COUNT(*) FILTER (WHERE privacy_status = 'SUPPRESSED'),
        COUNT(*) FILTER (
            WHERE voluntary_attrition_rate_pct IS NOT NULL
              AND voluntary_attrition_rate_pct < 0
        )
    FROM warehouse.m06_voluntary_attrition
""").fetchdf()

display(attrition_validation)

In [ ]:
warehouse_con.execute("""
-- M07: Gross pay by employee and month
CREATE OR REPLACE VIEW warehouse.m07_gross_pay AS
SELECT
    period_end_date,
    department_id,
    employee_id,
    ROUND(SUM(gross_pay), 2) AS gross_pay
FROM warehouse.clean_payroll
GROUP BY period_end_date, department_id, employee_id;

-- M08: Net pay by employee and month
CREATE OR REPLACE VIEW warehouse.m08_net_pay AS
SELECT
    period_end_date,
    department_id,
    employee_id,
    ROUND(SUM(net_pay), 2) AS net_pay
FROM warehouse.clean_payroll
GROUP BY period_end_date, department_id, employee_id;

-- M09: Employer payroll cost by employee and month
CREATE OR REPLACE VIEW warehouse.m09_employer_cost AS
SELECT
    period_end_date,
    department_id,
    employee_id,
    ROUND(SUM(employer_cost), 2) AS employer_cost
FROM warehouse.clean_payroll
GROUP BY period_end_date, department_id, employee_id;

-- M10: Cost per paid employee by department and month
CREATE OR REPLACE VIEW warehouse.m10_cost_per_head AS
SELECT
    period_end_date,
    department_id,
    COUNT(DISTINCT employee_id) AS employees_paid,
    ROUND(SUM(employer_cost) / NULLIF(COUNT(DISTINCT employee_id), 0), 2)
        AS cost_per_head
FROM warehouse.clean_payroll
GROUP BY period_end_date, department_id;

-- M11: Overtime share by department and month
CREATE OR REPLACE VIEW warehouse.m11_overtime_share AS
SELECT
    period_end_date,
    department_id,
    ROUND(
        100.0 * SUM(overtime_pay) / NULLIF(SUM(employer_cost), 0),
        2
    ) AS overtime_share_pct
FROM warehouse.clean_payroll
GROUP BY period_end_date, department_id;

-- M12: Variable pay share by department and month
CREATE OR REPLACE VIEW warehouse.m12_variable_pay_share AS
SELECT
    period_end_date,
    department_id,
    ROUND(
        100.0 * SUM(variable_pay) / NULLIF(SUM(gross_pay), 0),
        2
    ) AS variable_pay_share_pct
FROM warehouse.clean_payroll
GROUP BY period_end_date, department_id;
""")

print("Created M07–M12 payroll metric views.")

In [ ]:
warehouse_con.execute("""
-- M13: Attendance rate by department and month
CREATE OR REPLACE VIEW warehouse.m13_attendance_rate AS
WITH grouped AS (
    SELECT
        period_end_date,
        department_id,
        COUNT(DISTINCT employee_id) AS employee_count,
        SUM(present_days) AS present_days,
        SUM(half_days_unpaid) AS half_days_unpaid,
        SUM(scheduled_days) AS scheduled_days
    FROM warehouse.fact_attendance_monthly
    GROUP BY period_end_date, department_id
)
SELECT
    period_end_date,
    department_id,
    CASE
        WHEN employee_count >= 5
        THEN ROUND(
            100.0 * (present_days + 0.5 * half_days_unpaid)
            / NULLIF(scheduled_days, 0),
            2
        )
        ELSE NULL
    END AS attendance_rate_pct,
    CASE
        WHEN employee_count >= 5 THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM grouped;

-- M14: Late arrival rate by department and month
CREATE OR REPLACE VIEW warehouse.m14_late_arrival_rate AS
WITH grouped AS (
    SELECT
        period_end_date,
        department_id,
        COUNT(DISTINCT employee_id) AS employee_count,
        SUM(late_arrivals) AS late_arrivals,
        SUM(present_days) AS present_days
    FROM warehouse.fact_attendance_monthly
    GROUP BY period_end_date, department_id
)
SELECT
    period_end_date,
    department_id,
    CASE
        WHEN employee_count >= 5
        THEN ROUND(
            100.0 * late_arrivals / NULLIF(present_days, 0),
            2
        )
        ELSE NULL
    END AS late_arrival_rate_pct,
    CASE
        WHEN employee_count >= 5 THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM grouped;

-- M15: Leave without pay by employee and month (internal grain)
CREATE OR REPLACE VIEW warehouse.m15_lop_employee_month AS
SELECT
    period_end_date,
    department_id,
    employee_id,
    ROUND(SUM(lop_days), 2) AS lop_days
FROM warehouse.clean_payroll
GROUP BY period_end_date, department_id, employee_id;

-- M15: Privacy-safe department-month summary
CREATE OR REPLACE VIEW warehouse.m15_lop_department_month AS
WITH grouped AS (
    SELECT
        period_end_date,
        department_id,
        COUNT(DISTINCT employee_id) AS employee_count,
        SUM(lop_days) AS lop_days
    FROM warehouse.clean_payroll
    GROUP BY period_end_date, department_id
)
SELECT
    period_end_date,
    department_id,
    CASE
        WHEN employee_count >= 5 THEN ROUND(lop_days, 2)
        ELSE NULL
    END AS lop_days,
    CASE
        WHEN employee_count >= 5 THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM grouped;
""")

print("Created M13, M14, and M15 views.")

In [ ]:
warehouse_con.execute("""
-- M16: Revenue, excluding GST accounts
CREATE OR REPLACE VIEW warehouse.m16_revenue AS
SELECT
    CAST(date_trunc('month', entry_date) AS DATE) AS month_start_date,
    account_code,
    ROUND(-SUM(signed_amount_debit_positive), 2) AS revenue
FROM warehouse.fact_ledger
WHERE account_code BETWEEN 4001 AND 4004
GROUP BY 1, 2;

-- M17: Net profit, including account 4900 interest income
CREATE OR REPLACE VIEW warehouse.m17_net_profit AS
SELECT
    CAST(date_trunc('month', entry_date) AS DATE) AS month_start_date,
    ROUND(
        -SUM(signed_amount_debit_positive),
        2
    ) AS net_profit
FROM warehouse.fact_ledger
WHERE account_code BETWEEN 4000 AND 5999
GROUP BY 1;

-- M18: Operating margin
CREATE OR REPLACE VIEW warehouse.m18_operating_margin AS
WITH finance AS (
    SELECT
        CAST(date_trunc('month', entry_date) AS DATE) AS month_start_date,
        SUM(
            CASE
                WHEN account_code BETWEEN 4001 AND 4004
                THEN -signed_amount_debit_positive
                ELSE 0
            END
        ) AS revenue,
        SUM(
            CASE
                WHEN account_code BETWEEN 4000 AND 5999
                THEN -signed_amount_debit_positive
                ELSE 0
            END
        ) AS net_profit
    FROM warehouse.fact_ledger
    GROUP BY 1
)
SELECT
    month_start_date,
    ROUND(100.0 * net_profit / NULLIF(revenue, 0), 2)
        AS operating_margin_pct
FROM finance;

-- M19: Payroll as percentage of revenue.
-- Use ledger payroll accounts 5001-5005 for reconciliation.
CREATE OR REPLACE VIEW warehouse.m19_payroll_pct_revenue AS
WITH finance AS (
    SELECT
        CAST(date_trunc('month', entry_date) AS DATE) AS month_start_date,
        SUM(
            CASE
                WHEN account_code BETWEEN 5001 AND 5005
                THEN signed_amount_debit_positive
                ELSE 0
            END
        ) AS ledger_payroll_cost,
        SUM(
            CASE
                WHEN account_code BETWEEN 4001 AND 4004
                THEN -signed_amount_debit_positive
                ELSE 0
            END
        ) AS revenue
    FROM warehouse.fact_ledger
    GROUP BY 1
)
SELECT
    month_start_date,
    ROUND(
        100.0 * ledger_payroll_cost / NULLIF(revenue, 0),
        2
    ) AS payroll_pct_revenue
FROM finance;
""")

print("Created M16–M19 finance views.")

In [ ]:
invoice_check = warehouse_con.execute("""
    SELECT
        MIN(invoice_date) AS first_invoice_date,
        MAX(invoice_date) AS last_invoice_date,
        MIN(due_date) AS earliest_due_date,
        MAX(due_date) AS latest_due_date,
        COUNT(*) AS total_invoices,
        COUNT(*) FILTER (
            WHERE carried_in_from_prior_period = 'Y'
        ) AS carried_in_invoices
    FROM warehouse.fact_invoice
""").fetchdf()

print("Invoice date summary:")
display(invoice_check)

print("\nInvoice statuses:")
display(warehouse_con.execute("""
    SELECT
        status,
        COUNT(*) AS invoice_count
    FROM warehouse.fact_invoice
    GROUP BY status
    ORDER BY status
""").fetchdf())

In [ ]:
warehouse_con.execute("""
-- M20: Days sales outstanding, using the latest invoice date as cutoff.
-- Includes carried-in invoices and uses credit sales including GST.
CREATE OR REPLACE VIEW warehouse.m20_dso AS
WITH cutoff AS (
    SELECT MAX(invoice_date) AS report_date
    FROM warehouse.fact_invoice
),
receivables AS (
    SELECT
        c.report_date,
        SUM(i.balance_due) AS closing_receivables
    FROM cutoff c
    JOIN warehouse.fact_invoice i
      ON i.invoice_date <= c.report_date
    WHERE i.status NOT IN ('DRAFT', 'CANCELLED')
    GROUP BY c.report_date
),
sales_90d AS (
    SELECT
        c.report_date,
        SUM(i.total_amount) AS credit_sales_including_gst
    FROM cutoff c
    JOIN warehouse.fact_invoice i
      ON i.invoice_date > c.report_date - INTERVAL '90 days'
     AND i.invoice_date <= c.report_date
    WHERE i.status NOT IN ('DRAFT', 'CANCELLED')
    GROUP BY c.report_date
)
SELECT
    r.report_date,
    r.closing_receivables,
    s.credit_sales_including_gst,
    ROUND(
        90.0 * r.closing_receivables
        / NULLIF(s.credit_sales_including_gst, 0),
        2
    ) AS dso_days
FROM receivables r
JOIN sales_90d s USING (report_date);

-- M21: Overdue receivables by client at the reporting cutoff.
CREATE OR REPLACE VIEW warehouse.m21_overdue_receivables AS
WITH cutoff AS (
    SELECT MAX(invoice_date) AS report_date
    FROM warehouse.fact_invoice
)
SELECT
    c.report_date,
    i.client_id,
    ROUND(SUM(i.balance_due), 2) AS overdue_receivables
FROM cutoff c
JOIN warehouse.fact_invoice i
  ON i.due_date < c.report_date
WHERE i.status NOT IN ('DRAFT', 'CANCELLED')
  AND i.balance_due > 0
GROUP BY c.report_date, i.client_id;
""")

print("Created M20 DSO and M21 overdue receivables views.")

In [ ]:
print("M20 — Days Sales Outstanding")
display(warehouse_con.execute("""
    SELECT *
    FROM warehouse.m20_dso
""").fetchdf())

print("M21 — Overdue Receivables")
display(warehouse_con.execute("""
    SELECT
        COUNT(*) AS client_date_groups,
        SUM(overdue_receivables) AS total_overdue_receivables,
        COUNT(*) FILTER (
            WHERE overdue_receivables < 0
        ) AS negative_groups
    FROM warehouse.m21_overdue_receivables
""").fetchdf())

In [ ]:
warehouse_con.execute("""
-- M22: Task completion rate
CREATE OR REPLACE VIEW warehouse.m22_task_completion AS
WITH grouped AS (
    SELECT
        t.team_id,
        t.sprint_id,
        COUNT(*) FILTER (
            WHERE t.status <> 'Cancelled'
        ) AS planned_tasks,
        COUNT(*) FILTER (
            WHERE t.status = 'Done'
        ) AS completed_tasks
    FROM warehouse.fact_task t
    GROUP BY t.team_id, t.sprint_id
)
SELECT
    team_id,
    sprint_id,
    planned_tasks,
    completed_tasks,
    CASE
        WHEN planned_tasks >= 5
        THEN ROUND(100.0 * completed_tasks / NULLIF(planned_tasks, 0), 2)
        ELSE NULL
    END AS completion_rate_pct,
    CASE
        WHEN planned_tasks >= 5 THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM grouped;

-- M23: On-time delivery rate
CREATE OR REPLACE VIEW warehouse.m23_on_time_delivery AS
WITH grouped AS (
    SELECT
        team_id,
        sprint_id,
        COUNT(*) FILTER (
            WHERE status = 'Done'
        ) AS completed_tasks,
        COUNT(*) FILTER (
            WHERE status = 'Done'
              AND completed_on_time = 'Y'
        ) AS on_time_tasks
    FROM warehouse.fact_task
    GROUP BY team_id, sprint_id
)
SELECT
    team_id,
    sprint_id,
    completed_tasks,
    on_time_tasks,
    CASE
        WHEN completed_tasks >= 5
        THEN ROUND(100.0 * on_time_tasks / NULLIF(completed_tasks, 0), 2)
        ELSE NULL
    END AS on_time_rate_pct,
    CASE
        WHEN completed_tasks >= 5 THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM grouped;

-- M24: Velocity (story points completed)
CREATE OR REPLACE VIEW warehouse.m24_velocity AS
SELECT
    team_id,
    sprint_id,
    SUM(story_points) FILTER (
        WHERE status = 'Done'
    ) AS completed_story_points
FROM warehouse.fact_task
GROUP BY team_id, sprint_id;

-- M25: Average cycle time for completed tasks
CREATE OR REPLACE VIEW warehouse.m25_average_cycle_time AS
WITH grouped AS (
    SELECT
        team_id,
        sprint_id,
        COUNT(*) FILTER (
            WHERE status = 'Done'
              AND started_date IS NOT NULL
              AND completed_date IS NOT NULL
              AND completed_date >= started_date
        ) AS valid_completed_tasks,
        COUNT(*) FILTER (
            WHERE status = 'Done'
              AND started_date IS NOT NULL
              AND completed_date IS NOT NULL
              AND completed_date < started_date
        ) AS negative_cycle_time_errors,
        AVG(date_diff('day', started_date, completed_date)) FILTER (
            WHERE status = 'Done'
              AND started_date IS NOT NULL
              AND completed_date IS NOT NULL
              AND completed_date >= started_date
        ) AS avg_cycle_time_days
    FROM warehouse.fact_task
    GROUP BY team_id, sprint_id
)
SELECT
    team_id,
    sprint_id,
    valid_completed_tasks,
    negative_cycle_time_errors,
    CASE
        WHEN valid_completed_tasks >= 5
        THEN ROUND(avg_cycle_time_days, 2)
        ELSE NULL
    END AS avg_cycle_time_days,
    CASE
        WHEN valid_completed_tasks >= 5 THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM grouped;
""")

print("Created M22–M25 task metric views.")

In [ ]:
print("M26 — Anomaly flag values")
display(warehouse_con.execute("""
    SELECT flagged, COUNT(*) AS record_count
    FROM warehouse.fact_anomaly
    GROUP BY flagged
    ORDER BY flagged
""").fetchdf())

print("M28 — Forecast metric names")
display(warehouse_con.execute("""
    SELECT metric, COUNT(*) AS record_count
    FROM warehouse.fact_forecast
    GROUP BY metric
    ORDER BY metric
""").fetchdf())

print("M27 — Attrition risk bands")
display(warehouse_con.execute("""
    SELECT risk_band, COUNT(*) AS record_count
    FROM warehouse.fact_attrition_risk
    GROUP BY risk_band
    ORDER BY risk_band
""").fetchdf())

In [ ]:
warehouse_con.execute("""
-- M26: Anomaly flag rate by source and month
CREATE OR REPLACE VIEW warehouse.m26_anomaly_flag_rate AS
WITH grouped AS (
    SELECT
        source_dataset,
        CAST(date_trunc('month', period) AS DATE) AS month_start_date,
        COUNT(*) AS scored_records,
        COUNT(*) FILTER (WHERE flagged = 'Y') AS flagged_records
    FROM warehouse.fact_anomaly
    GROUP BY source_dataset, CAST(date_trunc('month', period) AS DATE)
)
SELECT
    source_dataset,
    month_start_date,
    scored_records,
    flagged_records,
    ROUND(
        100.0 * flagged_records / NULLIF(scored_records, 0),
        2
    ) AS anomaly_flag_rate_pct
FROM grouped;

-- M27: High-risk share by department.
-- Publish the group result only when at least 10 employees are represented.
CREATE OR REPLACE VIEW warehouse.m27_high_risk_share AS
WITH grouped AS (
    SELECT
        department_id,
        COUNT(DISTINCT employee_id) AS employee_count,
        COUNT(DISTINCT employee_id) FILTER (
            WHERE risk_band = 'High'
        ) AS high_risk_employees
    FROM warehouse.fact_attrition_risk
    GROUP BY department_id
)
SELECT
    department_id,
    employee_count,
    CASE
        WHEN employee_count >= 10
        THEN ROUND(
            100.0 * high_risk_employees / NULLIF(employee_count, 0),
            2
        )
        ELSE NULL
    END AS high_risk_share_pct,
    CASE
        WHEN employee_count >= 10 THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM grouped;

-- M28: Headcount forecast error (MAPE) by department and month.
-- Forecast and actual headcount are matched on department and month.
CREATE OR REPLACE VIEW warehouse.m28_forecast_error_mape AS
WITH actuals AS (
    SELECT
        month_start_date,
        department_id,
        headcount_end AS actual_headcount
    FROM warehouse.fact_headcount_monthly
),
matched AS (
    SELECT
        f.forecast_month AS month_start_date,
        f.department_id,
        a.actual_headcount,
        f.forecast_value AS forecast_headcount
    FROM warehouse.fact_forecast f
    JOIN actuals a
      ON a.month_start_date = f.forecast_month
     AND a.department_id = f.department_id
    WHERE f.metric = 'headcount_end'
      AND a.actual_headcount > 0
)
SELECT
    month_start_date,
    department_id,
    COUNT(*) AS matched_records,
    ROUND(
        AVG(
            ABS(actual_headcount - forecast_headcount)
            / NULLIF(actual_headcount, 0)
        ) * 100,
        2
    ) AS mape_pct
FROM matched
GROUP BY month_start_date, department_id;
""")

print("Created M26, M27, and M28.")

In [ ]:
warehouse_con.execute("""
CREATE OR REPLACE VIEW warehouse.metric_catalog AS
SELECT
    metric_id,
    metric_name,
    business_definition,
    formula,
    source_table,
    grain,
    time_period,
    owner_team,
    unit,
    good_direction,
    caveat,
    minimum_group_size
FROM read_csv_auto(
    '03_Data/csv/metric_dictionary.csv',
    header = TRUE
)
""")

print("Metric catalog created successfully.")

display(warehouse_con.execute("""
    SELECT
        COUNT(*) AS catalog_rows,
        COUNT(DISTINCT metric_id) AS unique_metric_ids
    FROM warehouse.metric_catalog
""").fetchdf())

In [ ]:
metric_view_names = [
    "m01_headcount",
    "m02_hires",
    "m03_exits",
    "m04_attrition_monthly",
    "m05_attrition_rolling_12m",
    "m06_voluntary_attrition",
    "m07_gross_pay",
    "m08_net_pay",
    "m09_employer_cost",
    "m10_cost_per_head",
    "m11_overtime_share",
    "m12_variable_pay_share",
    "m13_attendance_rate",
    "m14_late_arrival_rate",
    "m15_lop_employee_month",
    "m16_revenue",
    "m17_net_profit",
    "m18_operating_margin",
    "m19_payroll_pct_revenue",
    "m20_dso",
    "m21_overdue_receivables",
    "m22_task_completion",
    "m23_on_time_delivery",
    "m24_velocity",
    "m25_average_cycle_time",
    "m26_anomaly_flag_rate",
    "m27_high_risk_share",
    "m28_forecast_error_mape",
]

for view_name in metric_view_names:
    count = warehouse_con.execute(
        f"SELECT COUNT(*) FROM warehouse.{view_name}"
    ).fetchone()[0]
    print(f"{view_name}: {count} rows")

In [ ]:
forecast_date_check = warehouse_con.execute("""
    SELECT
        f.forecast_month,
        f.department_id,
        f.forecast_value,
        h.month_start_date AS actual_month,
        h.headcount_end AS actual_headcount
    FROM warehouse.fact_forecast f
    LEFT JOIN warehouse.fact_headcount_monthly h
      ON h.department_id = f.department_id
     AND h.month_start_date = f.forecast_month
    WHERE f.metric = 'headcount_end'
    ORDER BY f.forecast_month, f.department_id
    LIMIT 20
""").fetchdf()

display(forecast_date_check)

In [ ]:
forecast_coverage = warehouse_con.execute("""
    SELECT
        f.metric,
        COUNT(*) AS forecast_rows,
        COUNT(h.month_start_date) AS matched_actual_rows,
        COUNT(*) - COUNT(h.month_start_date) AS unmatched_rows
    FROM warehouse.fact_forecast f
    LEFT JOIN warehouse.fact_headcount_monthly h
      ON h.department_id = f.department_id
     AND h.month_start_date = f.forecast_month
    WHERE f.metric = 'headcount_end'
      AND f.department_id <> 'ALL'
    GROUP BY f.metric
""").fetchdf()

display(forecast_coverage)

print("Available actual months:")
display(warehouse_con.execute("""
    SELECT
        MIN(month_start_date) AS first_actual_month,
        MAX(month_start_date) AS last_actual_month,
        COUNT(DISTINCT month_start_date) AS actual_month_count
    FROM warehouse.fact_headcount_monthly
""").fetchdf())

In [ ]:
warehouse_con.execute("""
CREATE OR REPLACE VIEW warehouse.m28_forecast_error_mape AS
WITH actuals AS (
    SELECT
        month_start_date,
        department_id,
        headcount_end AS actual_headcount
    FROM warehouse.fact_headcount_monthly
),
department_forecasts AS (
    SELECT
        f.forecast_month AS month_start_date,
        f.department_id,
        f.forecast_value AS forecast_headcount,
        a.actual_headcount
    FROM warehouse.fact_forecast f
    LEFT JOIN actuals a
      ON a.month_start_date = f.forecast_month
     AND a.department_id = f.department_id
    WHERE f.metric = 'headcount_end'
      AND f.department_id <> 'ALL'
),
grouped AS (
    SELECT
        month_start_date,
        department_id,
        COUNT(*) AS forecast_records,
        COUNT(actual_headcount) AS matched_actual_records,
        AVG(
            ABS(actual_headcount - forecast_headcount)
            / NULLIF(actual_headcount, 0)
        ) FILTER (
            WHERE actual_headcount > 0
        ) * 100 AS mape_pct
    FROM department_forecasts
    GROUP BY month_start_date, department_id
)
SELECT
    month_start_date,
    department_id,
    forecast_records,
    matched_actual_records,
    ROUND(mape_pct, 2) AS mape_pct,
    CASE
        WHEN matched_actual_records = 0 THEN 'AWAITING_ACTUALS'
        ELSE 'CALCULATED'
    END AS calculation_status
FROM grouped;
""")

print("M28 updated.")

display(warehouse_con.execute("""
    SELECT
        calculation_status,
        COUNT(*) AS department_month_groups
    FROM warehouse.m28_forecast_error_mape
    GROUP BY calculation_status
    ORDER BY calculation_status
""").fetchdf())

In [ ]:
payroll_privacy_check = warehouse_con.execute("""
SELECT 'M07 Gross Pay' AS metric,
       COUNT(*) AS total_rows,
       COUNT(DISTINCT department_id) AS departments
FROM warehouse.m07_gross_pay

UNION ALL

SELECT 'M08 Net Pay',
       COUNT(*),
       COUNT(DISTINCT department_id)
FROM warehouse.m08_net_pay

UNION ALL

SELECT 'M09 Employer Cost',
       COUNT(*),
       COUNT(DISTINCT department_id)
FROM warehouse.m09_employer_cost
""").fetchdf()

display(payroll_privacy_check)

In [ ]:
warehouse_con.execute("""
CREATE OR REPLACE VIEW warehouse.m07_gross_pay_governed AS
SELECT
    period_end_date,
    department_id,
    COUNT(DISTINCT employee_id) AS employee_count,
    CASE
        WHEN COUNT(DISTINCT employee_id) >= 5
        THEN ROUND(SUM(gross_pay), 2)
        ELSE NULL
    END AS gross_pay,
    CASE
        WHEN COUNT(DISTINCT employee_id) >= 5
        THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM warehouse.clean_payroll
GROUP BY period_end_date, department_id;

CREATE OR REPLACE VIEW warehouse.m08_net_pay_governed AS
SELECT
    period_end_date,
    department_id,
    COUNT(DISTINCT employee_id) AS employee_count,
    CASE
        WHEN COUNT(DISTINCT employee_id) >= 5
        THEN ROUND(SUM(net_pay), 2)
        ELSE NULL
    END AS net_pay,
    CASE
        WHEN COUNT(DISTINCT employee_id) >= 5
        THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM warehouse.clean_payroll
GROUP BY period_end_date, department_id;

CREATE OR REPLACE VIEW warehouse.m09_employer_cost_governed AS
SELECT
    period_end_date,
    department_id,
    COUNT(DISTINCT employee_id) AS employee_count,
    CASE
        WHEN COUNT(DISTINCT employee_id) >= 5
        THEN ROUND(SUM(employer_cost), 2)
        ELSE NULL
    END AS employer_cost,
    CASE
        WHEN COUNT(DISTINCT employee_id) >= 5
        THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM warehouse.clean_payroll
GROUP BY period_end_date, department_id;
""")

print("Created governed payroll views: M07, M08, M09.")

In [ ]:
payroll_governance_check = warehouse_con.execute("""
SELECT 'M07 Gross Pay' AS metric,
       privacy_status,
       COUNT(*) AS group_count,
       SUM(CASE WHEN employee_count < 5 THEN 1 ELSE 0 END)
           AS groups_below_threshold,
       SUM(CASE WHEN employee_count < 5 AND gross_pay IS NOT NULL
                THEN 1 ELSE 0 END) AS unsafe_published_groups
FROM warehouse.m07_gross_pay_governed
GROUP BY privacy_status

UNION ALL

SELECT 'M08 Net Pay',
       privacy_status,
       COUNT(*),
       SUM(CASE WHEN employee_count < 5 THEN 1 ELSE 0 END),
       SUM(CASE WHEN employee_count < 5 AND net_pay IS NOT NULL
                THEN 1 ELSE 0 END)
FROM warehouse.m08_net_pay_governed
GROUP BY privacy_status

UNION ALL

SELECT 'M09 Employer Cost',
       privacy_status,
       COUNT(*),
       SUM(CASE WHEN employee_count < 5 THEN 1 ELSE 0 END),
       SUM(CASE WHEN employee_count < 5 AND employer_cost IS NOT NULL
                THEN 1 ELSE 0 END)
FROM warehouse.m09_employer_cost_governed
GROUP BY privacy_status

ORDER BY metric, privacy_status
""").fetchdf()

display(payroll_governance_check)

In [ ]:
for metric_name, view_name, value_column in [
    ("M02 Hires", "m02_hires", "hires"),
    ("M03 Exits", "m03_exits", "exits"),
]:
    result = warehouse_con.execute(f"""
        SELECT
            COUNT(*) AS total_groups,
            SUM(CASE WHEN {value_column} < 0 THEN 1 ELSE 0 END)
                AS negative_values
        FROM warehouse.{view_name}
    """).fetchdf()

    print(metric_name)
    display(result)

In [ ]:
warehouse_con.execute("""
CREATE OR REPLACE VIEW warehouse.m02_hires_governed AS
SELECT
    h.month_start_date,
    h.department_id,
    d.department_name,
    h.hires,
    CASE
        WHEN hc.headcount_end >= 5 THEN h.hires
        ELSE NULL
    END AS hires_published,
    CASE
        WHEN hc.headcount_end >= 5 THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM warehouse.m02_hires h
LEFT JOIN warehouse.fact_headcount_monthly hc
    ON h.month_start_date = hc.month_start_date
   AND h.department_id = hc.department_id
LEFT JOIN warehouse.dim_department d
    ON h.department_id = d.department_id;

CREATE OR REPLACE VIEW warehouse.m03_exits_governed AS
SELECT
    e.month_start_date,
    e.department_id,
    d.department_name,
    e.exits,
    CASE
        WHEN hc.headcount_end >= 5 THEN e.exits
        ELSE NULL
    END AS exits_published,
    CASE
        WHEN hc.headcount_end >= 5 THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM warehouse.m03_exits e
LEFT JOIN warehouse.fact_headcount_monthly hc
    ON e.month_start_date = hc.month_start_date
   AND e.department_id = hc.department_id
LEFT JOIN warehouse.dim_department d
    ON e.department_id = d.department_id;
""")

print("Created governed views for M02 Hires and M03 Exits.")

In [ ]:
for metric, view_name, value_column in [
    ("M02 Hires", "m02_hires_governed", "hires_published"),
    ("M03 Exits", "m03_exits_governed", "exits_published"),
]:
    print(metric)

    result = warehouse_con.execute(f"""
        SELECT
            privacy_status,
            COUNT(*) AS group_count,
            SUM(CASE
                WHEN privacy_status = 'SUPPRESSED'
                 AND {value_column} IS NOT NULL
                THEN 1 ELSE 0
            END) AS unsafe_published_groups
        FROM warehouse.{view_name}
        GROUP BY privacy_status
        ORDER BY privacy_status
    """).fetchdf()

    display(result)

In [ ]:
employee_count_check = warehouse_con.execute("""
WITH monthly_groups AS (
    SELECT
        h.month_start_date,
        h.department_id,
        h.hires,
        h.exits,
        COUNT(DISTINCT CASE
            WHEN e.hire_date >= h.month_start_date
             AND e.hire_date < h.month_start_date + INTERVAL '1 month'
            THEN e.employee_id
        END) AS actual_hires,
        COUNT(DISTINCT CASE
            WHEN e.exit_date >= h.month_start_date
             AND e.exit_date < h.month_start_date + INTERVAL '1 month'
            THEN e.employee_id
        END) AS actual_exits
    FROM warehouse.fact_headcount_monthly h
    LEFT JOIN warehouse.dim_employee e
      ON e.department_id = h.department_id
     AND (
          (e.hire_date >= h.month_start_date
           AND e.hire_date < h.month_start_date + INTERVAL '1 month')
          OR
          (e.exit_date >= h.month_start_date
           AND e.exit_date < h.month_start_date + INTERVAL '1 month')
     )
    GROUP BY h.month_start_date, h.department_id, h.hires, h.exits
)
SELECT
    COUNT(*) AS total_groups,
    SUM(CASE WHEN actual_hires < 5 THEN 1 ELSE 0 END)
        AS hire_groups_below_5,
    SUM(CASE WHEN actual_exits < 5 THEN 1 ELSE 0 END)
        AS exit_groups_below_5,
    SUM(CASE WHEN actual_hires <> hires THEN 1 ELSE 0 END)
        AS hire_count_mismatches,
    SUM(CASE WHEN actual_exits <> exits THEN 1 ELSE 0 END)
        AS exit_count_mismatches
FROM monthly_groups
""").fetchdf()

display(employee_count_check)

In [ ]:
source_count_check = warehouse_con.execute("""
SELECT
    month_start_date,
    department_id,
    hires,
    exits,
    headcount_start,
    headcount_end
FROM warehouse.fact_headcount_monthly
ORDER BY month_start_date, department_id
LIMIT 15
""").fetchdf()

display(source_count_check)

In [ ]:
headcount_reconciliation = warehouse_con.execute("""
SELECT
    COUNT(*) AS total_groups,
    SUM(CASE
        WHEN headcount_start + hires - exits
             = headcount_end
        THEN 1 ELSE 0
    END) AS reconciled_groups,
    SUM(CASE
        WHEN headcount_start + hires - exits
             <> headcount_end
        THEN 1 ELSE 0
    END) AS mismatched_groups
FROM warehouse.fact_headcount_monthly
""").fetchdf()

display(headcount_reconciliation)

In [ ]:
display(warehouse_con.execute("""
SELECT
    month_start_date,
    department_id,
    headcount_start,
    hires,
    exits,
    transfers_in,
    transfers_out,
    headcount_end,
    headcount_start + hires - exits
        AS calculated_headcount_before_transfers,
    headcount_end -
        (headcount_start + hires - exits)
        AS difference
FROM warehouse.fact_headcount_monthly
WHERE headcount_start + hires - exits <> headcount_end
ORDER BY month_start_date, department_id
LIMIT 30
""").fetchdf())

In [ ]:
transfer_reconciliation = warehouse_con.execute("""
SELECT
    COUNT(*) AS total_groups,
    SUM(CASE
        WHEN headcount_start + hires - exits
             + transfers_in - transfers_out = headcount_end
        THEN 1 ELSE 0
    END) AS reconciled_groups,
    SUM(CASE
        WHEN headcount_start + hires - exits
             + transfers_in - transfers_out <> headcount_end
        THEN 1 ELSE 0
    END) AS mismatched_groups
FROM warehouse.fact_headcount_monthly
""").fetchdf()

display(transfer_reconciliation)

In [ ]:
display(warehouse_con.execute("""
SELECT
    COUNT(*) AS total_groups,
    COUNT(*) FILTER (
        WHERE headcount_start + hires - exits
              + transfers_in - transfers_out = headcount_end
    ) AS reconciled_groups,
    COUNT(*) FILTER (
        WHERE headcount_start + hires - exits
              + transfers_in - transfers_out <> headcount_end
    ) AS mismatched_groups
FROM warehouse.fact_headcount_monthly
""").fetchdf())

In [ ]:
catalog_check = warehouse_con.execute("""
SELECT
    COUNT(*) AS total_metrics,
    COUNT(DISTINCT metric_id) AS unique_metric_ids
FROM warehouse.metric_catalog
""").fetchdf()

print("Metric catalog check:")
display(catalog_check)

view_check = warehouse_con.execute("""
SELECT
    table_name
FROM information_schema.views
WHERE table_schema = 'warehouse'
  AND (
      table_name LIKE 'm%'
      OR table_name = 'metric_catalog'
  )
ORDER BY table_name
""").fetchdf()

print("Metric views currently available:")
display(view_check)

In [ ]:
m04_manual_check = warehouse_con.execute("""
SELECT
    month_start_date,
    department_id,
    headcount_start,
    exits,
    ROUND(
        100.0 * exits / NULLIF(headcount_start, 0),
        2
    ) AS manually_calculated_attrition_pct,
    attrition_pct AS view_attrition_pct,
    privacy_status
FROM warehouse.m04_attrition_monthly
WHERE headcount_start >= 10
  AND headcount_start > 0
ORDER BY month_start_date, department_id
LIMIT 10
""").fetchdf()

display(m04_manual_check)

In [ ]:
m04_manual_check = warehouse_con.execute("""
SELECT
    v.month_start_date,
    v.department_id,
    h.headcount_start,
    h.exits,
    ROUND(
        100.0 * h.exits / NULLIF(h.headcount_start, 0),
        2
    ) AS manually_calculated_attrition_pct,
    v.attrition_rate_pct AS view_attrition_pct,
    v.privacy_status
FROM warehouse.m04_attrition_monthly v
JOIN warehouse.fact_headcount_monthly h
  ON v.month_start_date = h.month_start_date
 AND v.department_id = h.department_id
WHERE h.headcount_start >= 10
ORDER BY v.month_start_date, v.department_id
LIMIT 10
""").fetchdf()

display(m04_manual_check)

In [ ]:
m10_manual_check = warehouse_con.execute("""
WITH manual_calc AS (
    SELECT
        period_end_date,
        department_id,
        COUNT(DISTINCT employee_id) AS employees_paid,
        SUM(employer_cost) AS total_employer_cost,
        ROUND(
            SUM(employer_cost)
            / NULLIF(COUNT(DISTINCT employee_id), 0),
            2
        ) AS manually_calculated_cost_per_head
    FROM warehouse.clean_payroll
    GROUP BY period_end_date, department_id
)
SELECT
    m.period_end_date,
    m.department_id,
    m.employees_paid,
    ROUND(m.total_employer_cost, 2) AS total_employer_cost,
    m.manually_calculated_cost_per_head,
    v.cost_per_head AS view_cost_per_head
FROM manual_calc m
JOIN warehouse.m10_cost_per_head v
  ON m.period_end_date = v.period_end_date
 AND m.department_id = v.department_id
ORDER BY m.period_end_date, m.department_id
LIMIT 10
""").fetchdf()

display(m10_manual_check)

In [ ]:
m11_manual_check = warehouse_con.execute("""
WITH manual_calc AS (
    SELECT
        period_end_date,
        department_id,
        COUNT(DISTINCT employee_id) AS employees_paid,
        SUM(overtime_pay) AS total_overtime_pay,
        SUM(employer_cost) AS total_employer_cost,
        ROUND(
            100.0 * SUM(overtime_pay)
            / NULLIF(SUM(employer_cost), 0),
            2
        ) AS manually_calculated_overtime_share_pct
    FROM warehouse.clean_payroll
    GROUP BY period_end_date, department_id
)
SELECT
    m.period_end_date,
    m.department_id,
    m.employees_paid,
    ROUND(m.total_overtime_pay, 2) AS total_overtime_pay,
    ROUND(m.total_employer_cost, 2) AS total_employer_cost,
    m.manually_calculated_overtime_share_pct,
    v.overtime_share_pct AS view_overtime_share_pct
FROM manual_calc m
JOIN warehouse.m11_overtime_share v
  ON m.period_end_date = v.period_end_date
 AND m.department_id = v.department_id
ORDER BY m.period_end_date, m.department_id
LIMIT 10
""").fetchdf()

display(m11_manual_check)

In [ ]:
warehouse_con.execute("""
CREATE OR REPLACE VIEW warehouse.m11_overtime_share_governed AS
SELECT
    period_end_date,
    department_id,
    COUNT(DISTINCT employee_id) AS employees_paid,
    CASE
        WHEN COUNT(DISTINCT employee_id) >= 5
        THEN ROUND(
            100.0 * SUM(overtime_pay)
            / NULLIF(SUM(employer_cost), 0),
            2
        )
        ELSE NULL
    END AS overtime_share_pct,
    CASE
        WHEN COUNT(DISTINCT employee_id) >= 5
        THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM warehouse.clean_payroll
GROUP BY period_end_date, department_id;
""")

print("Created privacy-governed M11 view.")

In [ ]:
m11_privacy_check = warehouse_con.execute("""
SELECT
    privacy_status,
    COUNT(*) AS group_count,
    SUM(CASE
        WHEN employees_paid < 5
         AND overtime_share_pct IS NOT NULL
        THEN 1 ELSE 0
    END) AS unsafe_published_groups
FROM warehouse.m11_overtime_share_governed
GROUP BY privacy_status
ORDER BY privacy_status
""").fetchdf()

display(m11_privacy_check)

In [ ]:
m13_manual_check = warehouse_con.execute("""
WITH manual_calc AS (
    SELECT
        period_end_date,
        department_id,
        COUNT(DISTINCT employee_id) AS employee_count,
        SUM(present_days) AS total_present_days,
        SUM(half_days_unpaid) AS total_half_days_unpaid,
        SUM(scheduled_days) AS total_scheduled_days,
        ROUND(
            100.0 * (
                SUM(present_days)
                + 0.5 * SUM(half_days_unpaid)
            ) / NULLIF(SUM(scheduled_days), 0),
            2
        ) AS manually_calculated_attendance_pct
    FROM warehouse.fact_attendance_monthly
    GROUP BY period_end_date, department_id
)
SELECT
    m.period_end_date,
    m.department_id,
    m.employee_count,
    m.total_present_days,
    m.total_half_days_unpaid,
    m.total_scheduled_days,
    m.manually_calculated_attendance_pct,
    v.attendance_rate_pct AS view_attendance_pct
FROM manual_calc m
JOIN warehouse.m13_attendance_rate v
  ON m.period_end_date = v.period_end_date
 AND m.department_id = v.department_id
ORDER BY m.period_end_date, m.department_id
LIMIT 10
""").fetchdf()

display(m13_manual_check)

In [ ]:
warehouse_con.execute("""
CREATE OR REPLACE VIEW warehouse.m13_attendance_rate_governed AS
SELECT
    period_end_date,
    department_id,
    COUNT(DISTINCT employee_id) AS employee_count,
    SUM(scheduled_days) AS total_scheduled_days,
    CASE
        WHEN COUNT(DISTINCT employee_id) >= 5
         AND SUM(scheduled_days) > 0
        THEN ROUND(
            100.0 * (
                SUM(present_days)
                + 0.5 * SUM(half_days_unpaid)
            ) / NULLIF(SUM(scheduled_days), 0),
            2
        )
        ELSE NULL
    END AS attendance_rate_pct,
    CASE
        WHEN COUNT(DISTINCT employee_id) < 5
          OR SUM(scheduled_days) <= 0
        THEN 'SUPPRESSED'
        ELSE 'PUBLISHED'
    END AS privacy_status
FROM warehouse.fact_attendance_monthly
GROUP BY period_end_date, department_id;
""")

print("Created privacy-governed M13 attendance view.")

In [ ]:
m13_privacy_check = warehouse_con.execute("""
SELECT
    privacy_status,
    COUNT(*) AS group_count,
    SUM(CASE
        WHEN (
            employee_count < 5
            OR total_scheduled_days <= 0
        )
        AND attendance_rate_pct IS NOT NULL
        THEN 1 ELSE 0
    END) AS unsafe_published_groups
FROM warehouse.m13_attendance_rate_governed
GROUP BY privacy_status
ORDER BY privacy_status
""").fetchdf()

display(m13_privacy_check)

In [ ]:
m16_manual_check = warehouse_con.execute("""
WITH manual_calc AS (
    SELECT
        d.month_start_date,
        l.account_code,
        ROUND(-SUM(l.signed_amount_debit_positive), 2)
            AS manually_calculated_revenue
    FROM warehouse.fact_ledger l
    JOIN warehouse.dim_date d
      ON l.date_key = d.date_key
    WHERE l.account_code BETWEEN 4001 AND 4004
    GROUP BY d.month_start_date, l.account_code
)
SELECT
    m.month_start_date,
    m.account_code,
    m.manually_calculated_revenue,
    v.revenue AS view_revenue
FROM manual_calc m
JOIN warehouse.m16_revenue v
  ON m.month_start_date = v.month_start_date
 AND m.account_code = v.account_code
ORDER BY m.month_start_date, m.account_code
LIMIT 10
""").fetchdf()

display(m16_manual_check)

In [ ]:
final_task4_check = warehouse_con.execute("""
SELECT
    (SELECT COUNT(*)
     FROM warehouse.metric_catalog) AS catalog_metrics,

    (SELECT COUNT(DISTINCT metric_id)
     FROM warehouse.metric_catalog) AS unique_metric_ids,

    (SELECT COUNT(*)
     FROM information_schema.views
     WHERE table_schema = 'warehouse'
       AND table_name IN (
           'm01_headcount_governed',
           'm02_hires_governed',
           'm03_exits_governed',
           'm07_gross_pay_governed',
           'm08_net_pay_governed',
           'm09_employer_cost_governed',
           'm11_overtime_share_governed',
           'm13_attendance_rate_governed',
           'metric_catalog'
       )) AS expected_governed_views_present
""").fetchdf()

display(final_task4_check)

In [ ]:
print("Metric catalog row count:")
print(
    warehouse_con.execute("""
        SELECT COUNT(*) 
        FROM warehouse.metric_catalog
    """).fetchone()[0]
)

print("\nMetric catalog sample:")
display(
    warehouse_con.execute("""
        SELECT metric_id, metric_name
        FROM warehouse.metric_catalog
        ORDER BY metric_id
        LIMIT 10
    """).fetchdf()
)

In [ ]:
display(warehouse_con.execute("""
SELECT
    employee_id,
    department_id,
    hire_date,
    exit_date,
    employment_status
FROM warehouse.dim_employee
ORDER BY hire_date
LIMIT 15
""").fetchdf())

In [ ]:
employee_duplicates_check = warehouse_con.execute("""
SELECT
    employee_id,
    COUNT(*) AS record_count,
    COUNT(DISTINCT department_id) AS department_count,
    COUNT(DISTINCT hire_date) AS hire_date_count,
    COUNT(DISTINCT exit_date) AS exit_date_count
FROM warehouse.dim_employee
GROUP BY employee_id
HAVING COUNT(*) > 1
ORDER BY record_count DESC, employee_id
LIMIT 20
""").fetchdf()

display(employee_duplicates_check)

In [ ]:
employee_history_check = warehouse_con.execute("""
SELECT
    employee_id,
    department_id,
    hire_date,
    exit_date,
    valid_from,
    valid_to,
    is_current
FROM warehouse.dim_employee
WHERE employee_id IN ('EMP-0002', 'EMP-0003', 'EMP-0051')
ORDER BY employee_id, valid_from
""").fetchdf()

display(employee_history_check)

In [ ]:
historical_event_check = warehouse_con.execute("""
WITH employee_events AS (
    SELECT DISTINCT
        employee_id,
        department_id,
        hire_date,
        exit_date
    FROM warehouse.dim_employee
),
monthly_counts AS (
    SELECT
        h.month_start_date,
        h.department_id,
        h.hires,
        h.exits,
        COUNT(DISTINCT CASE
            WHEN e.hire_date >= h.month_start_date
             AND e.hire_date < h.month_start_date + INTERVAL '1 month'
            THEN e.employee_id
        END) AS distinct_hires,
        COUNT(DISTINCT CASE
            WHEN e.exit_date >= h.month_start_date
             AND e.exit_date < h.month_start_date + INTERVAL '1 month'
            THEN e.employee_id
        END) AS distinct_exits
    FROM warehouse.fact_headcount_monthly h
    LEFT JOIN employee_events e
      ON e.department_id = h.department_id
    GROUP BY
        h.month_start_date,
        h.department_id,
        h.hires,
        h.exits
)
SELECT
    COUNT(*) AS department_month_groups,
    SUM(CASE WHEN distinct_hires < 5 THEN 1 ELSE 0 END)
        AS hire_groups_below_5,
    SUM(CASE WHEN distinct_exits < 5 THEN 1 ELSE 0 END)
        AS exit_groups_below_5,
    SUM(CASE WHEN distinct_hires <> hires THEN 1 ELSE 0 END)
        AS hire_count_mismatches,
    SUM(CASE WHEN distinct_exits <> exits THEN 1 ELSE 0 END)
        AS exit_count_mismatches
FROM monthly_counts
""").fetchdf()

display(historical_event_check)

In [ ]:
print("Monthly headcount rows:")
print(
    warehouse_con.execute("""
        SELECT COUNT(*)
        FROM warehouse.fact_headcount_monthly
    """).fetchone()[0]
)

print("\nMonthly headcount sample:")
display(
    warehouse_con.execute("""
        SELECT
            month_start_date,
            department_id,
            hires,
            exits,
            transfers_in,
            transfers_out,
            headcount_start,
            headcount_end
        FROM warehouse.fact_headcount_monthly
        ORDER BY month_start_date, department_id
        LIMIT 10
    """).fetchdf()
)

In [ ]:
display(warehouse_con.execute("""
SELECT
    CASE
        WHEN headcount_start >= 5
         AND headcount_end >= 5
        THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS proposed_privacy_status,
    COUNT(*) AS department_month_groups
FROM warehouse.fact_headcount_monthly
GROUP BY 1
ORDER BY 1
""").fetchdf())

In [ ]:
warehouse_con.execute("""
CREATE OR REPLACE VIEW warehouse.m02_hires_governed AS
SELECT
    h.month_start_date,
    h.department_id,
    d.department_name,
    h.hires,
    CASE
        WHEN hc.headcount_start >= 5
         AND hc.headcount_end >= 5
        THEN h.hires
        ELSE NULL
    END AS hires_published,
    CASE
        WHEN hc.headcount_start >= 5
         AND hc.headcount_end >= 5
        THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM warehouse.m02_hires h
JOIN warehouse.fact_headcount_monthly hc
  ON h.month_start_date = hc.month_start_date
 AND h.department_id = hc.department_id
LEFT JOIN warehouse.dim_department d
  ON h.department_id = d.department_id;

CREATE OR REPLACE VIEW warehouse.m03_exits_governed AS
SELECT
    e.month_start_date,
    e.department_id,
    d.department_name,
    e.exits,
    CASE
        WHEN hc.headcount_start >= 5
         AND hc.headcount_end >= 5
        THEN e.exits
        ELSE NULL
    END AS exits_published,
    CASE
        WHEN hc.headcount_start >= 5
         AND hc.headcount_end >= 5
        THEN 'PUBLISHED'
        ELSE 'SUPPRESSED'
    END AS privacy_status
FROM warehouse.m03_exits e
JOIN warehouse.fact_headcount_monthly hc
  ON e.month_start_date = hc.month_start_date
 AND e.department_id = hc.department_id
LEFT JOIN warehouse.dim_department d
  ON e.department_id = d.department_id;
""")

print("Updated M02 and M03 governed views.")

In [ ]:
for metric, view_name, value_column in [
    ("M02 Hires", "m02_hires_governed", "hires_published"),
    ("M03 Exits", "m03_exits_governed", "exits_published"),
]:
    print(metric)

    display(warehouse_con.execute(f"""
        SELECT
            privacy_status,
            COUNT(*) AS group_count,
            SUM(CASE
                WHEN (
                    hc.headcount_start < 5
                    OR hc.headcount_end < 5
                ) AND v.{value_column} IS NOT NULL
                THEN 1 ELSE 0
            END) AS unsafe_published_groups
        FROM warehouse.{view_name} v
        JOIN warehouse.fact_headcount_monthly hc
          ON v.month_start_date = hc.month_start_date
         AND v.department_id = hc.department_id
        GROUP BY privacy_status
        ORDER BY privacy_status
    """).fetchdf())

In [ ]:
metric_views = [
    "m01_headcount", "m02_hires", "m03_exits",
    "m04_attrition_monthly", "m05_attrition_rolling_12m",
    "m06_voluntary_attrition", "m07_gross_pay", "m08_net_pay",
    "m09_employer_cost", "m10_cost_per_head",
    "m11_overtime_share", "m12_variable_pay_share",
    "m13_attendance_rate", "m14_late_arrival_rate",
    "m15_lop_employee_month", "m16_revenue",
    "m17_net_profit", "m18_operating_margin",
    "m19_payroll_pct_revenue", "m20_dso",
    "m21_overdue_receivables", "m22_task_completion",
    "m23_on_time_delivery", "m24_velocity",
    "m25_average_cycle_time", "m26_anomaly_flag_rate",
    "m27_high_risk_share", "m28_forecast_error_mape"
]

inventory = []

for view_name in metric_views:
    row_count = warehouse_con.execute(
        f"SELECT COUNT(*) FROM warehouse.{view_name}"
    ).fetchone()[0]

    inventory.append({
        "metric_view": view_name,
        "row_count": row_count,
        "status": "HAS DATA" if row_count > 0 else "REVIEW"
    })

inventory_df = pd.DataFrame(inventory)
display(inventory_df)

print("Official metric views:", len(inventory_df))
print("Views returning rows:", (inventory_df["row_count"] > 0).sum())
print("Views needing review:", (inventory_df["row_count"] == 0).sum())

In [ ]:
# Final governed-view and catalog persistence check
required_views = [
    "metric_catalog",
    "m01_headcount_governed",
    "m02_hires_governed",
    "m03_exits_governed",
    "m07_gross_pay_governed",
    "m08_net_pay_governed",
    "m09_employer_cost_governed",
    "m11_overtime_share_governed",
    "m13_attendance_rate_governed",
]

for view_name in required_views:
    row = warehouse_con.execute(f"""
        SELECT COUNT(*)
        FROM warehouse.{view_name}
    """).fetchone()

    print(f"{view_name}: {row[0]} rows")

In [ ]:
from pathlib import Path

print("Project folder:", project_path)
print("Notebook working directory:", Path.cwd())
print("Warehouse file:", warehouse_db_path)
print("Warehouse file exists:", warehouse_db_path.exists())

print("\nMetric dictionary exists:",
      (project_path / "03_Data" / "csv" / "metric_dictionary.csv").exists())

print("Starter SQL exists:",
      (project_path / "05_Starter" / "metric_views_examples.sql").exists())

In [ ]:
print(con.execute("""
    SELECT table_schema, table_name, table_type
    FROM information_schema.tables
    WHERE table_name = 'metric_catalog'
    ORDER BY table_schema
""").df())

In [ ]:
import duckdb
from pathlib import Path

db_path = Path(
    r"C:\Users\Hp\Documents\pulseiq\05_Starter\pulseiq_warehouse.duckdb"
)

print("Database exists:", db_path.exists())

con = duckdb.connect(str(db_path), read_only=True)

print("Connected successfully!")

In [ ]:
print(con.execute("""
    SELECT table_schema, table_name, table_type
    FROM information_schema.tables
    WHERE table_name = 'metric_catalog'
    ORDER BY table_schema
""").df())

In [ ]:
metric_df = con.execute("""
    SELECT *
    FROM warehouse.metric_catalog
""").df()

print("Metric catalog shape:", metric_df.shape)
print("Columns:", metric_df.columns.tolist())

display(metric_df)

In [ ]:
print("Rows:", len(metric_df))
print("\nMissing values per column:")
display(metric_df.isna().sum().to_frame("missing_count"))

In [ ]:
from pathlib import Path

project_folder = Path(r"C:\Users\Hp\Documents\pulseiq")

catalog_path = project_folder / "Task4_Metric_Catalog.csv"
metric_df.to_csv(catalog_path, index=False)

print("Metric count:", len(metric_df))
print("Saved:", catalog_path)
print("File exists:", catalog_path.exists())

In [ ]:
for table in [
    "fact_payroll",
    "fact_ledger",
    "fact_task"
]:
    print(f"\n--- {table} ---")
    print(con.execute(
        f"DESCRIBE warehouse.{table}"
    ).df()[["column_name", "column_type"]])

In [ ]:
import pandas as pd

# 1. Gross pay — M07
gross_pay = con.execute("""
    SELECT SUM(gross_pay) AS gross_pay_total
    FROM warehouse.fact_payroll
""").df()

# 2. Net pay — M08
net_pay = con.execute("""
    SELECT SUM(net_pay) AS net_pay_total
    FROM warehouse.fact_payroll
""").df()

# 3. Revenue inputs — M16
revenue_check = con.execute("""
    SELECT
        COUNT(*) AS ledger_rows,
        SUM(signed_amount_debit_positive) AS signed_total,
        SUM(CASE
            WHEN account_code BETWEEN 4001 AND 4004
            THEN signed_amount_debit_positive
            ELSE 0
        END) AS revenue_account_signed_total
    FROM warehouse.fact_ledger
""").df()

# 4. Net profit inputs — M17
profit_check = con.execute("""
    SELECT
        SUM(CASE
            WHEN account_code BETWEEN 4000 AND 4999
            THEN signed_amount_debit_positive
            ELSE 0
        END) AS account_4xxx_total,
        SUM(CASE
            WHEN account_code BETWEEN 5000 AND 5999
            THEN signed_amount_debit_positive
            ELSE 0
        END) AS account_5xxx_total
    FROM warehouse.fact_ledger
""").df()

# 5. Task completion rate — M22
task_check = con.execute("""
    SELECT
        COUNT(*) AS total_tasks,
        SUM(CASE WHEN status = 'Done' THEN 1 ELSE 0 END)
            AS done_tasks,
        SUM(CASE WHEN status != 'Cancelled' THEN 1 ELSE 0 END)
            AS non_cancelled_tasks,
        100.0 * SUM(CASE WHEN status = 'Done' THEN 1 ELSE 0 END)
        / NULLIF(
            SUM(CASE WHEN status != 'Cancelled' THEN 1 ELSE 0 END),
            0
        ) AS completion_rate_pct
    FROM warehouse.fact_task
""").df()

print("M07 — Gross Pay")
display(gross_pay)

print("M08 — Net Pay")
display(net_pay)

print("M16 — Revenue inputs (confirm sign convention)")
display(revenue_check)

print("M17 — Net Profit inputs (confirm sign convention)")
display(profit_check)

print("M22 — Task Completion Rate")
display(task_check)

In [ ]:
display(con.execute("""
    SELECT
        account_code,
        COUNT(*) AS row_count,
        SUM(debit) AS total_debit,
        SUM(credit) AS total_credit,
        SUM(signed_amount_debit_positive) AS signed_total
    FROM warehouse.fact_ledger
    WHERE account_code BETWEEN 4001 AND 5005
    GROUP BY account_code
    ORDER BY account_code
""").df())

In [ ]:
display(con.execute("""
    SELECT
        -SUM(CASE
            WHEN account_code BETWEEN 4001 AND 4004
            THEN signed_amount_debit_positive
            ELSE 0
        END) AS revenue,

        -SUM(CASE
            WHEN account_code BETWEEN 4000 AND 4999
            THEN signed_amount_debit_positive
            ELSE 0
        END)
        - SUM(CASE
            WHEN account_code BETWEEN 5000 AND 5999
            THEN signed_amount_debit_positive
            ELSE 0
        END) AS net_profit

    FROM warehouse.fact_ledger
""").df())

In [ ]:
import pandas as pd
from pathlib import Path

# Calculate metrics directly from the warehouse
checks = con.execute("""
    SELECT 'M07' AS metric_id,
           'Gross pay' AS metric_name,
           SUM(gross_pay) AS calculated_value
    FROM warehouse.fact_payroll

    UNION ALL

    SELECT 'M08', 'Net pay', SUM(net_pay)
    FROM warehouse.fact_payroll

    UNION ALL

    SELECT 'M16', 'Revenue',
           -SUM(CASE
               WHEN account_code BETWEEN 4001 AND 4004
               THEN signed_amount_debit_positive
               ELSE 0
           END)
    FROM warehouse.fact_ledger

    UNION ALL

    SELECT 'M17', 'Net profit',
           -SUM(CASE
               WHEN account_code BETWEEN 4000 AND 4999
               THEN signed_amount_debit_positive
               ELSE 0
           END)
           - SUM(CASE
               WHEN account_code BETWEEN 5000 AND 5999
               THEN signed_amount_debit_positive
               ELSE 0
           END)
    FROM warehouse.fact_ledger

    UNION ALL

    SELECT 'M22', 'Task completion rate',
           100.0 * SUM(CASE WHEN status = 'Done' THEN 1 ELSE 0 END)
           / NULLIF(
               SUM(CASE WHEN status != 'Cancelled' THEN 1 ELSE 0 END),
               0
           )
    FROM warehouse.fact_task
""").df()

# Add definitions and hand-check notes
notes = {
    "M07": "SUM(gross_pay) from fact_payroll",
    "M08": "SUM(net_pay) from fact_payroll",
    "M16": "Negative signed total for ledger accounts 4001–4004",
    "M17": "Negative signed total for 4xxx accounts minus positive signed total for 5xxx accounts",
    "M22": "Done tasks / non-cancelled tasks × 100"
}

checks["formula_checked"] = checks["metric_id"].map(notes)
checks["validation_note"] = (
    "Calculated from warehouse; compare with catalog definition"
)

output_path = Path(
    r"C:\Users\Hp\Documents\pulseiq\Task4_Metric_Hand_Checks.csv"
)

checks.to_csv(output_path, index=False)

display(checks)
print("Saved:", output_path)
print("File exists:", output_path.exists())